# Six-month collections forecast: Adam vs GRU vs LSTM

This notebook compares a feed-forward MLP trained with Adam, a GRU trained with Adam, and an LSTM trained with Adam. A 7-day mean baseline is included. All models use the same Golden Payments series, 28-day lookback, chronological holdout and forecast horizon.

**Important:** the supplied payment history is 1 Jan–8 Aug 2026 (220 daily observations); August is partial. This does not provide 12 complete months or a full annual seasonal cycle. Missing months are not fabricated. Forecasts are experimental gross-payment scenarios, not incremental recovery or ROI.

## 1. Run the reproducible pipeline
The script loads the Golden Payments output, builds daily totals, trains each model, evaluates the last 28 observed days, and produces six monthly forecasts from September 2026 through February 2027.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'src' / 'forecast_model_comparison.py').exists():
    ROOT = ROOT / 'collections_analytics_project'
subprocess.run([sys.executable, str(ROOT / 'src' / 'forecast_model_comparison.py')], cwd=ROOT, check=True)


## 2. Compare out-of-sample performance
MAE and RMSE are in rupees per day. WMAPE is scale-normalized. Lower is better. The holdout is chronological, not randomly shuffled.

In [ ]:
import pandas as pd
metrics = pd.read_csv(ROOT / 'outputs' / 'forecast_model_comparison_metrics.csv')
metrics


## 3. Compare six-month projections
The monthly outputs are recursive model forecasts. Differences between architectures should be interpreted cautiously because the sample is short and no external portfolio/staffing/cost drivers are available.

In [ ]:
forecast = pd.read_csv(ROOT / 'outputs' / 'six_month_model_monthly_forecasts.csv')
forecast.assign(forecast_recovery_cr=forecast.forecast_recovery_rupees/1e7).pivot(index='month', columns='model', values='forecast_recovery_cr').round(2)


## 4. Model selection rule
Choose the model only if it consistently beats the naive baseline across multiple rolling-origin backtests and produces stable forecasts. A single 28-day holdout is preliminary evidence, not sufficient production validation. The current data does not support a reliable 12-month training history. Obtain the missing historical period and repeat evaluation before using these numbers in the ₹10 Cr investment case.